# Ungraded Lab: Text Generation and the KV Cache

A walkthrough of how an autoregressive language model produces text one token at a time, and how a single optimization, the KV cache, speeds up the generation loop.


In the graded notebook for this module you will measure latency, throughput, and memory on real models. Before that, it is useful to look at what happens inside a single call to `model(...)` and a single step of the generation loop.

You will:

1. Load a small LLM (GPT-2) from HuggingFace and inspect its architecture.
2. Generate one next token by hand from the model's output logits.
3. Run the naïve generation loop, re-feeding the full sequence at every step, and measure how the per-token latency grows.
4. Add a KV cache so that each step processes only the new token, and compare the two timing curves.

Nothing in this notebook is graded.

> **Why GPT-2?** It is small, fast on CPU, and uses the same decoder-only autoregressive structure as modern chat models. Latency-sensitive products such as autocomplete or inline code suggestions still use GPT-2-class models in production because they can deliver tens of milliseconds per token. The optimization patterns shown here (prefill / decode separation, KV caching) are the same ones used at much larger scale.
>
> There is also a pedagogical reason. In newer architectures (Llama, Qwen, Mistral) KV caching is so deeply integrated into the HuggingFace implementation that you cannot easily run them without it. GPT-2 still lets you call the model with no cache, returns `past_key_values` only when asked, and lets you wire the loop yourself. That makes it a good model for building a KV cache by hand instead of toggling a flag.


## Setup

Import the libraries used in this lab. `transformers` provides the model and tokenizer, `torch` is the tensor backend, and `matplotlib` is used at the end to plot the per-token latency curves.


In [1]:
import matplotlib.pyplot as plt
import numpy as np
import time
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

/usr/local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


---

## 1. Loading an LLM from HuggingFace


The model weights and tokenizer are loaded from a local copy of GPT-2 in `./models/gpt2`. `AutoModelForCausalLM` selects the causal (decoder-only) variant, which is the one that produces text one token at a time.


In [2]:
model_name = "./models/gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

Print the model to see its architecture.

In [3]:
print(model)

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)


---

## 2. Generating one token from model outputs

Instead of calling `model.generate()`, we will do one step by hand. The recipe is:

1. Tokenize the prompt.
2. Run a forward pass to get logits.
3. Pick the next token from those logits.
4. Decode the token id back to text.

### Tokenize the prompt

The tokenizer maps text to integer token ids. `return_tensors="pt"` asks for PyTorch tensors.


In [4]:
prompt = "The quick brown fox jumped over the"
inputs = tokenizer(prompt, return_tensors="pt")
inputs

{'input_ids': tensor([[  464,  2068,  7586, 21831, 11687,   625,   262]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1]])}

The output contains:

- `input_ids`: the integer ids the tokenizer produced for the prompt.
- `attention_mask`: all ones for now. It tells the model which positions are real tokens versus padding.

### Forward pass

Wrap the call in `torch.no_grad()` because this is inference. There is no need to build the gradient graph or pay for the memory it would use.


In [5]:
with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits
print(logits.shape)

torch.Size([1, 7, 50257])


The logits tensor has shape `[batch, sequence_length, vocab_size]`. For our single-prompt input that is `[1, 7, 50257]`: one batch element, seven input tokens, and one score per vocabulary entry.

During training the model needs a prediction at every position. During inference we only care about the next token, so we keep just the last position along the sequence axis (`logits[0, -1, :]`) and take the `argmax` over the vocabulary to get the most likely next token id.


In [6]:
last_logits = logits[0, -1, :]
next_token_id = last_logits.argmax()
next_token_id


tensor(13990)

### Decode the token

Translate the chosen token id back into a string.


In [7]:
tokenizer.decode(next_token_id)

' fence'

### Look at the alternatives

`argmax` is **greedy decoding**: always pick the single most likely token. To see what other choices the model considered, look at the top-10 candidates by logit value.


In [8]:
top_k = torch.topk(last_logits, k=10)
tokens = [tokenizer.decode(tk) for tk in top_k.indices]
tokens

[' fence',
 ' edge',
 ' railing',
 ' wall',
 ' table',
 ' tree',
 ' top',
 ' counter',
 ' ground',
 ' side']

### Append the new token to the input

To generate a second token, the simplest strategy is to take the token we just produced, concatenate it onto `input_ids`, extend the attention mask by one, and run the model again on the new (longer) sequence.


In [9]:
next_inputs = {
    "input_ids": torch.cat(
        [inputs["input_ids"], next_token_id.reshape((1, 1))],
        dim=1
    ),
    "attention_mask": torch.cat(
        [inputs["attention_mask"], torch.tensor([[1]])],
        dim=1
    ),
}

In [10]:
print(next_inputs["input_ids"],
      next_inputs["input_ids"].shape)
print(next_inputs["attention_mask"],
      next_inputs["attention_mask"].shape)

tensor([[  464,  2068,  7586, 21831, 11687,   625,   262, 13990]]) torch.Size([1, 8])
tensor([[1, 1, 1, 1, 1, 1, 1, 1]]) torch.Size([1, 8])


---

## 3. Prefill and decode: optimizing token generation over multiple steps

Doing one token at a time by hand is fine. To generate a sentence we need to repeat the process in a loop, and once we do that the cost of generation becomes visible. This section first runs the naïve loop (re-feeding the entire sequence at every step) and then introduces the KV cache to fix the performance problem it exposes.


### A helper for one generation step

Wrap the four-step recipe (forward pass, last logits, argmax, return id) into a single function so the loop below stays short.


In [11]:
def generate_token(inputs):
    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits
    last_logits = logits[0, -1, :]
    next_token_id = last_logits.argmax()
    return next_token_id

### The naïve generation loop

Generate 30 tokens. At each step:

- Time the call to `generate_token`.
- Concatenate the new token onto `input_ids` and extend the attention mask.
- Decode the new id and store the string for inspection at the end.

After the loop runs, look at the total time and the list of generated tokens to confirm the output is sensible.


In [12]:
generated_tokens = []
next_inputs = inputs
durations_s = []
for _ in range(30):
    t0 = time.time()
    next_token_id = generate_token(next_inputs)
    durations_s += [time.time() - t0]
    
    next_inputs = {
        "input_ids": torch.cat(
            [next_inputs["input_ids"], next_token_id.reshape((1, 1))],
            dim=1),
        "attention_mask": torch.cat(
            [next_inputs["attention_mask"], torch.tensor([[1]])],
            dim=1),
    }
    
    next_token = tokenizer.decode(next_token_id)
    generated_tokens.append(next_token)

print(f"{sum(durations_s)} s")
print(generated_tokens)

1.2036988735198975 s
[' fence', ' and', ' ran', ' to', ' the', ' other', ' side', ' of', ' the', ' fence', '.', ' He', ' was', ' about', ' to', ' run', ' when', ' he', ' saw', ' the', ' fox', '.', ' He', ' ran', ' to', ' the', ' other', ' side', ' of', ' the']


### Plot the per-token latency

Plot `durations_s` against the token index. The x-axis is the token number (0 through 9), the y-axis is the time it took to produce that single token in seconds.


**Note:** Exact numbers depend on your machine, but the overall shape will be similar.

The first token is usually a bit slower (cache warm-up at the framework or OS level), and after that the per-token time grows as more tokens are generated. The reason is that we are re-running the full attention computation over a sequence that gets longer at every step. With 200 tokens the last few would be visibly more expensive than the first few. This is the cost the next section removes.


In [ ]:
plt.plot(durations_s)
plt.show()

---

## 4. Speeding up generation with the KV cache

The biggest computational cost in a transformer is the attention computation. For each token, attention builds three matrices: query (`Q`), key (`K`), and value (`V`). The number of rows in `K` and `V` equals the sequence length, so attention cost grows with how much context the model is looking at.

When we generated token #2 above, we re-ran the model on the original prompt plus token #1. But the `K` and `V` rows for the original prompt had already been computed when we generated token #1. We just threw them away and recomputed them. Same story for token #3, #4, and so on.

The KV cache is the fix: store `K` and `V` for every past token, and on the next step only compute the new row and append it to the cache. Two phases:

- The first call ("prefill") runs the model on the full prompt and returns the cache.
- Every later call ("decode") feeds in only the single new token, plus the cache from the previous step. Attention reads the cached `K`/`V` for the past tokens and only computes new entries for the new token.

The split between prefill and decode is one of the core ideas in LLM inference. Most of the LLM-serving literature (paged attention, continuous batching, etc.) builds on top of it.

### A helper that returns the cache

`outputs.past_key_values` holds the cached `K`/`V` tensors. Update the helper from before so it returns both the next token id and the cache.


In [ ]:
def generate_token_with_past(inputs):
    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits
    last_logits = logits[0, -1, :]
    next_token_id = last_logits.argmax()
    return next_token_id, outputs.past_key_values

### The cached generation loop

Same loop as before, with two changes:

1. `input_ids` for the next step is just the new token, not the full sequence. The model already has everything else inside `past_key_values`.
2. We pass `past_key_values` back in.

The `attention_mask` still grows by one each step, because the model needs to know the total length of the sequence (cached + new) for masking purposes, even though `input_ids` only carries the new token.


In [ ]:
generated_tokens = []
next_inputs = inputs
durations_cached_s = []
for _ in range(30):
    t0 = time.time()
    next_token_id, past_key_values = \
        generate_token_with_past(next_inputs)
    durations_cached_s += [time.time() - t0]
    
    next_inputs = {
        "input_ids": next_token_id.reshape((1, 1)),
        "attention_mask": torch.cat(
            [next_inputs["attention_mask"], torch.tensor([[1]])],
            dim=1),
        "past_key_values": past_key_values,
    }
    
    next_token = tokenizer.decode(next_token_id)
    generated_tokens.append(next_token)

print(f"{sum(durations_cached_s)} s")
print(generated_tokens)

### Compare the two loops

Plot the cached and uncached durations on the same axes.


**Note:** Exact numbers depend on your machine, but the relative shape will be similar.

What you should see:

- **Token 0 (prefill):** both versions take roughly the same time. They are doing the same work, running the full prompt through the model.
- **Tokens 1+ (decode):** the cached version drops to a much smaller, roughly flat per-token cost. The naïve version drifts upward as the sequence gets longer.

The blue line is the naïve loop; the orange line is the cached loop. The gap between them is the redundant attention computation we removed. The longer the generation, the bigger the gap.

The KV cache makes single-stream generation fast, but it lives in GPU memory and grows with sequence length and batch size. The trade-off between fast decode and finite memory is what the graded notebook in this module measures. Optimizations like PagedAttention (vLLM), continuous batching, and quantized caches all exist to manage the KV cache more carefully than the simple version implemented here.


In [ ]:
plt.plot(durations_s)
plt.plot(durations_cached_s)
plt.show()